Modeling: Same Wavelength
=========================

This script fits a multiple `Imaging` datasets observed at the same wavelength of a 'galaxy-scale' strong lens with a
model where:

 - The lens galaxy's light is omitted (and is not present in the simulated data).
 - The lens galaxy's total mass distribution is an `Isothermal`; the external shear is an
   `ExternalShear` held in a `MassField`.
 - The source galaxy's light is an MGE.

This script demonstrates how PyAutoLens's multi-dataset modeling tools can also simultaneously analyse datasets
observed at the same wavelength.

An example use case might be analysing undithered HST images before they are combined via the multidrizzing process,
to remove correlated noise in the data.

This is an advanced script and assumes previous knowledge of the core **PyAutoLens** API for lens modeling. Thus,
certain parts of code are not documented to ensure the script is concise.

__Contents__

- **Pixel Scales:** If observed at the same wavelength, it is likely the datasets have the same pixel-scale.
- **Dataset & Mask:** Standard set up of the dataset and mask that is fitted.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **Model:** Compose the lens model fitted to the data.
- **Shared Source Mesh (Pixelization):** Reconstruct the source of every exposure on one shared Delaunay mesh via `shared_preloads=True`.
- **Search:** Configure the non-linear search used to fit the model.
- **Result:** Overview of the results of the model-fit.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autolens_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Pixel Scales__

If observed at the same wavelength, it is likely the datasets have the same pixel-scale.

Nevertheless, we specify this as a list as there could be an exception.

In [ ]:
pixel_scales_list = [0.1, 0.1]

__Dataset__

Load and plot each multi-wavelength strong lens dataset, using a list of their waveband colors.

In [ ]:
dataset_type = "multi_dataset"
dataset_label = "imaging"
dataset_name = "same_wavelength"

dataset_path = Path("dataset") / dataset_type / dataset_label / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/multi_dataset/features/same_wavelength/simulator.py"],
        check=True,
    )

dataset_list = [
    al.Imaging.from_fits(
        data_path=Path(dataset_path, f"image_{i}.fits"),
        psf_path=Path(dataset_path, f"psf_{i}.fits"),
        noise_map_path=Path(dataset_path, f"noise_map_{i}.fits"),
        pixel_scales=pixel_scales,
    )
    for i, pixel_scales in enumerate(pixel_scales_list)
]

for dataset in dataset_list:
    aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

Define a 3.0" circular mask, which includes the emission of the lens and source galaxies.

For multi-wavelength lens modeling, we use the same mask for every dataset whenever possible. This is not
absolutely necessary, but provides a more reliable analysis.

In [ ]:
mask_radius = 3.0

mask_list = [
    al.Mask2D.circular(
        shape_native=dataset.shape_native,
        pixel_scales=dataset.pixel_scales,
        radius=mask_radius,
    )
    for dataset in dataset_list
]

dataset_list = [
    dataset.apply_mask(mask=mask) for imaging, mask in zip(dataset_list, mask_list)
]

for dataset in dataset_list:
    aplt.subplot_imaging_dataset(dataset=dataset)

__Analysis__

We create an `Analysis` object for every dataset.

In [ ]:
analysis_list = [al.AnalysisImaging(dataset=dataset) for dataset in dataset_list]

__Model__

We compose a lens model where:

 - The lens galaxy's total mass distribution is an `Isothermal`; the external shear is an
   `ExternalShear` held in a `MassField` [7 parameters].

 - The source galaxy's light is an MGE with 1 x 20 Gaussians [4 parameters].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=11.

In [ ]:
# Lens:

mass = af.Model(al.mp.Isothermal)
lens = af.Model(al.Galaxy, redshift=0.5, mass=mass)

# Source:

bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=False,
)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

# Overall Lens Model:

field = af.Model(al.MassField, redshift=0.5, shear=af.Model(al.mp.ExternalShear))

model = af.Collection(
    galaxies=af.Collection(lens=lens, source=source),
    fields=field,
)

We now combine them using the factor analysis class, which allows us to fit the two datasets simultaneously.

Unlike other examples, no customization to the model is applied that, for example, adds more free parameters,
given that the datasets do not vary over wavelength.

In [ ]:
analysis_factor_list = []

for analysis in analysis_list:
    model_analysis = model.copy()

    analysis_factor = af.AnalysisFactor(prior_model=model_analysis, analysis=analysis)

    analysis_factor_list.append(analysis_factor)

factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

The `info` of the model shows us there are two models each with linear light profiles.

In [ ]:
print(factor_graph.global_prior_model.info)

The same model can also be visualized as a figure, making its structure easier to understand at a glance.

The figure shows how the model is organized: which parameters belong to each component, and whether they are free, 
fixed, shared, linked by an expression, solved during the fit, or not configured. `model.info` provides the 
corresponding numerical details, including the prior assigned to each free parameter and the value of each fixed 
parameter.

In [ ]:
af.ModelPlotter(factor_graph.global_prior_model).figure()

__Shared Source Mesh (Pixelization)__

When the source is reconstructed on a pixelization (see `features/pixelization`), fitting each exposure as an
independent factor has a subtle drawback: every exposure builds its own source-plane mesh (the image-mesh centres
are ray-traced independently per dataset), so the per-exposure source reconstructions live on slightly different
source-pixel grids and cannot be compared pixel-by-pixel.

Because all exposures share one lens model, the source-plane mesh is exposure-invariant. Setting
`shared_preloads=True` on every `AnalysisImaging` opts the factor graph into the shared-state mechanism: the lead
factor ray-traces its image-mesh once per likelihood evaluation and every exposure maps its own image grid onto
that identical shared Delaunay mesh:

    analysis_list = [
        al.AnalysisImaging(
            dataset=dataset,
            adapt_images=adapt_images,  # image-mesh pixelizations (e.g. `Overlay` + `Delaunay`) use adapt images
            shared_preloads=True,
        )
        for dataset in dataset_list
    ]

Every exposure then reconstructs the source on the same source-pixel grid: the reconstructions are directly
comparable (their differences diagnose registration errors and PSF quality), and the redundant per-exposure
image-mesh construction and mesh ray-tracing are skipped. Each exposure still builds its own mapping matrix,
PSF-blurred mapping matrix, curvature matrix and regularization matrix — these depend on the exposure's own PSF,
pixel offsets and (for adaptive schemes) data, so they are never shared.

If the exposures have known pixel offsets (e.g. undithered HST exposures), combine this with the per-dataset
`DatasetModel` offsets shown in `features/dataset_offsets` — the shared mesh is defined in the lead exposure's
frame and every other exposure's offset grid is traced onto it. Offsets from a data reduction (e.g. differences
of each frame's `target_pixel` in PyAutoReduce frame products) are best applied as fixed, known offsets; for
precision applications make the per-exposure (dy, dx) free parameters with Gaussian priors of width equal to the
measured registration residuals (typically ~0.1-0.3 pixels for HST/JWST).

Note that each exposure still receives its own reconstruction (solved against its own data). A single joint
reconstruction solved against all exposures simultaneously is a separate, planned feature.

__Search__

The model is fitted to the data using the nested sampling algorithm Nautilus (see `start.here.py` for a 
full description).

In [ ]:
search = af.Nautilus(
    path_prefix=Path("multi_dataset", "modeling"),
    name="same_wavelength",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

__Model-Fit__

In [ ]:
result_list = search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)

__Result__

The result object returned by this model-fit is a list of `Result` objects, because we used a factor graph.
Each result corresponds to each analysis, and therefore corresponds to the model-fit at that wavelength.

In [ ]:
print(result_list[0].max_log_likelihood_instance)
print(result_list[1].max_log_likelihood_instance)

Plotting each result's tracer shows that the source appears different, owning to its different intensities.

In [ ]:
for result in result_list:
    aplt.subplot_tracer(tracer=result.max_log_likelihood_tracer, grid=result.grids.lp)

    aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)

The `Samples` object still has the dimensions of the overall non-linear search (in this case N=11). 

Therefore, the samples is identical in every result object.

In [ ]:
aplt.corner_anesthetic(samples=result_list.samples)

Checkout `autolens_workspace/*/guides/results` for a full description of analysing results.